# Bipartite Agenda/Emphasis Network — Gephi Export

Produces `nodes.csv` and `edges.csv` for a **bipartite** graph in Gephi.  
Two node types: **(1) source sub-units** (news outlets / talkshow programmes / kamer document types) and **(2) meta-topics**.  
Edges connect a sub-unit to each meta-topic it covers, weighted by the **within-sub-unit share** of documents in that meta-topic (so each sub-unit's edge weights sum to ~1).

Based purely on document counts — no embeddings or cosine similarity.

**Kamer note:** `kamer` path is `None` until BERTopic + `topic_meta` mapping is run on that corpus.

In [1]:
import re
import pathlib
import pandas as pd

NB_DIR = pathlib.Path(".").resolve()

# ============================================================
# PARAMETERS — edit these before running
# ============================================================

SOURCES = {
    "news": {
        "path":        "../../news/final_df/cleaned_final_df.csv",
        "subunit_col": "outlet",      # one node per news outlet
        "topic_col":   "topic",
        "meta_col":    "topic_meta",
        "node_type":   "news",
        "id_prefix":   "news",         # node Ids: news_ad, news_vk, …
    },
    "talkshows": {
        "path":        "../../subtitles/data/subs_labelled_mapped.csv",
        "subunit_col": "program",     # one node per talkshow programme
        "topic_col":   "topic",
        "meta_col":    "topic_meta",
        "node_type":   "talkshows",
        "id_prefix":   "show",         # node Ids: show_jinek, show_pauw, …
    },
    "kamer": {
        "path":        None,           # TODO: set once BERTopic + topic_meta mapping is done
        "subunit_col": "type",        # one node per document type (beleidsnota / vergaderstuk / plenair verslag)
        "topic_col":   "topic",
        "meta_col":    "topic_meta",
        "node_type":   "kamer",
        "id_prefix":   "kamer",        # node Ids: kamer_beleidsnota, …
    },
}

MIN_DOCS        = 0    # drop sub-units with fewer total (non-outlier) documents than this
MIN_EDGE_WEIGHT = 0.0  # drop edges whose within-sub-unit share is below this threshold

OUT_NODES = "nodes.csv"
OUT_EDGES = "edges.csv"
# ============================================================

## 1. Load source data

In [2]:
def slugify(s):
    """Lowercase alphanumeric slug, spaces/punctuation → underscore."""
    s = str(s).lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


frames = {}

for src_name, cfg in SOURCES.items():
    if cfg["path"] is None:
        print(f"[SKIP] {src_name}: path is None")
        continue
    abs_path = (NB_DIR / cfg["path"]).resolve()
    if not abs_path.exists():
        print(f"[SKIP] {src_name}: file not found at {abs_path}")
        continue
    df = pd.read_csv(abs_path)
    required = [cfg["subunit_col"], cfg["topic_col"], cfg["meta_col"]]
    missing  = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{src_name}: missing required columns {missing}")
    before = len(df)
    df = df[df[cfg["topic_col"]] != -1].copy()
    df[cfg["meta_col"]]    = df[cfg["meta_col"]].fillna("UNKNOWN").astype(str).str.strip()
    df[cfg["subunit_col"]] = df[cfg["subunit_col"]].fillna("UNKNOWN").astype(str).str.strip()
    frames[src_name] = df
    print(
        f"[OK] {src_name}: {len(df):,} docs "
        f"({before - len(df)} outlier rows dropped), "
        f"{df[cfg['subunit_col']].nunique()} sub-units, "
        f"{df[cfg['meta_col']].nunique()} meta-topics"
    )

if not frames:
    raise RuntimeError("No source files were loaded. Check SOURCES paths above.")

[OK] news: 13,209 docs (0 outlier rows dropped), 8 sub-units, 18 meta-topics
[OK] talkshows: 495 docs (0 outlier rows dropped), 9 sub-units, 12 meta-topics
[SKIP] kamer: path is None


## 2. Build nodes table

Two node types:
- **Source sub-unit nodes** — one per (arena, sub-unit); `size` = document count in that sub-unit.
- **Meta-topic nodes** — one per distinct meta-topic across all sources; `size` = total docs assigned to that meta-topic.

In [3]:
subunit_rows    = []   # node dicts for sub-units
subunit_records = []   # (node_id, group_df, meta_col) used when building edges

for src_name, cfg in SOURCES.items():
    if src_name not in frames:
        continue
    df     = frames[src_name]
    prefix = cfg["id_prefix"]
    for subunit, grp in df.groupby(cfg["subunit_col"]):
        if len(grp) < MIN_DOCS:
            continue
        node_id = f"{prefix}_{slugify(subunit)}"
        subunit_rows.append({
            "Id":        node_id,
            "Label":     subunit,
            "node_type": cfg["node_type"],
            "size":      len(grp),
        })
        subunit_records.append((node_id, grp.copy(), cfg["meta_col"]))

# Meta-topic nodes: size = total docs across all loaded sources
all_meta = pd.concat(
    [frames[s][SOURCES[s]["meta_col"]] for s in frames],
    ignore_index=True,
)
meta_counts = all_meta.value_counts()

meta_rows = [
    {
        "Id":        f"meta_{slugify(meta)}",
        "Label":     meta,
        "node_type": "metatopic",
        "size":      int(count),
    }
    for meta, count in meta_counts.items()
]

nodes_df = pd.DataFrame(subunit_rows + meta_rows)

dupes = nodes_df[nodes_df.duplicated("Id", keep=False)]
if len(dupes):
    print("WARNING — duplicate node Ids detected:", dupes["Id"].tolist())

print(f"Total nodes: {len(nodes_df)}  "
      f"({len(subunit_rows)} sub-units + {len(meta_rows)} meta-topics)")
nodes_df

Total nodes: 35  (17 sub-units + 18 meta-topics)


,Id,Label,node_type,size
0,news_ad,AD,news,3595
1,news_ga,GA,news,401
2,news_nrc,NRC,news,2071
3,news_nu_nl,NU.nl,news,1076
4,news_parool,Parool,news,448
5,news_tg,TG,news,2297
6,news_tr,TR,news,898
7,news_vk,VK,news,2423
8,show_bar_laat,bar laat,talkshows,21
9,show_caf_kockelmann,café kockelmann,talkshows,14


## 3. Build edges table

**Weight** = within-sub-unit share: for a given sub-unit, the fraction of its documents that belong to a given meta-topic.  
By construction each sub-unit's edge weights sum to 1 (before any `min_edge_weight` filter).

In [4]:
# Lookup: meta-topic label → node Id
meta_id = {
    row["Label"]: row["Id"]
    for _, row in nodes_df[nodes_df["node_type"] == "metatopic"].iterrows()
}

edge_rows = []

for node_id, grp, meta_col in subunit_records:
    total = len(grp)
    for meta, count in grp[meta_col].value_counts().items():
        weight = count / total
        if weight < MIN_EDGE_WEIGHT:
            continue
        target = meta_id.get(meta)
        if target is None:
            print(f"WARNING: meta-topic '{meta}' not in node table — skipping edge from {node_id}")
            continue
        edge_rows.append({
            "Source": node_id,
            "Target": target,
            "Weight": round(weight, 6),
            "Type":   "Undirected",
        })

edges_df = pd.DataFrame(edge_rows)
print(f"Total edges: {len(edges_df)}")
edges_df

Total edges: 209


,Source,Target,Weight,Type
0,news_ad,meta_media_arts_culture_sports,0.194993,Undirected
1,news_ad,meta_consumer_tech_products,0.114047,Undirected
2,news_ad,meta_society,0.080946,Undirected
3,news_ad,meta_business_finance,0.078999,Undirected
4,news_ad,meta_science_health,0.077051,Undirected
...,...,...,...,...
204,show_pauw,meta_ai_for_security,0.105263,Undirected
205,show_pauw,meta_science_health,0.105263,Undirected
206,show_pauw,meta_social_media,0.105263,Undirected
207,show_pauw,meta_ai_warfare_milirtary_conflicts,0.052632,Undirected


## 4. Diagnostic summary

In [5]:
print("=" * 60)
print("DIAGNOSTIC SUMMARY")
print()

print("SUB-UNIT COUNTS PER ARENA:")
for src_name, cfg in SOURCES.items():
    arena_nodes = nodes_df[nodes_df["node_type"] == cfg["node_type"]]
    if src_name not in frames:
        print(f"  {src_name}: [skipped — no data loaded]")
    else:
        print(f"  {src_name}: {len(arena_nodes)} sub-unit nodes")
        for _, row in arena_nodes.sort_values("size", ascending=False).iterrows():
            print(f"      {row['Label']:<35s}  {row['size']:>6,} docs")
print(f"  metatopic: {len(nodes_df[nodes_df['node_type'] == 'metatopic'])} nodes")
print()

print(f"TOTAL EDGES: {len(edges_df)}")
print()

print("WEIGHT SUM CHECK (should be ~1.0 for every sub-unit):")
if len(edges_df):
    weight_sums = edges_df.groupby("Source")["Weight"].sum()
    bad = weight_sums[(weight_sums - 1.0).abs() > 0.01]
    if len(bad):
        print("  WARNING — sub-units deviating from 1.0:")
        for nid, s in bad.items():
            print(f"    {nid}: {s:.6f}")
    else:
        print(f"  All {len(weight_sums)} sub-units sum to ~1.0  ✓")
        print(f"  Range: [{weight_sums.min():.6f}, {weight_sums.max():.6f}]")
else:
    print("  (no edges to check)")
print("=" * 60)

DIAGNOSTIC SUMMARY

SUB-UNIT COUNTS PER ARENA:
  news: 8 sub-unit nodes
      AD                                    3,595 docs
      VK                                    2,423 docs
      TG                                    2,297 docs
      NRC                                   2,071 docs
      NU.nl                                 1,076 docs
      TR                                      898 docs
      Parool                                  448 docs
      GA                                      401 docs
  talkshows: 9 sub-unit nodes
      goedemorgen nederland                   296 docs
      de wereld draait door                    70 docs
      eva                                      42 docs
      jinek                                    23 docs
      bar laat                                 21 docs
      pauw                                     19 docs
      café kockelmann                          14 docs
      de vooravond                              6 docs
      m           

## 5. Export

In [6]:
nodes_df.to_csv(OUT_NODES, index=False)
edges_df.to_csv(OUT_EDGES, index=False)

print(f"Saved {len(nodes_df)} nodes  -> {OUT_NODES}")
print(f"Saved {len(edges_df)} edges  -> {OUT_EDGES}")
print()
print("Node type breakdown:")
print(nodes_df["node_type"].value_counts().to_string())

Saved 35 nodes  -> nodes.csv
Saved 209 edges  -> edges.csv

Node type breakdown:
node_type
metatopic    18
talkshows     9
news          8
